In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import sklearn
import xgboost
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

In [ ]:
# Use relative paths so the notebook can be moved to another machine.
FEATURE_ARCHIVE = Path('outputs/dinov2_features.npz')
PROBABILITY_OUTPUT_DIR = Path('outputs/base_probabilities')
FIXED_FOLD_DIR = Path('data/5_Fold_CV_Splits')
USE_SAVED_FOLDS = True
FOLD_ID_COLUMN = 'ImageName'
OUTER_FOLDS = 5
INNER_FOLDS = 5
RANDOM_STATE = 42
N_ESTIMATORS = 100
MAX_DEPTH = 6
LEARNING_RATE = 0.30
N_JOBS = -1

In [ ]:
MODALITIES = ('rgb', 'tir', 'dem')
CLASS_NAMES = (
    'Background',
    'Debris Accumulation',
    'Exposed Rock Mass',
    'Agricultural Encroachment',
    'Drainage Infrastructure',
)
FEATURE_DIMENSION = 384

def load_feature_archive(path: Path):
    with np.load(path, allow_pickle=False) as archive:
        sample_ids = archive['sample_ids'].astype(str)
        labels = archive['labels'].astype(np.int64)
        features = {modality: archive[modality].astype(np.float32) for modality in MODALITIES}
    if len(labels) != len(sample_ids):
        raise ValueError('sample_ids and labels have different lengths.')
    if not np.array_equal(np.unique(labels), np.arange(len(CLASS_NAMES))):
        raise ValueError('Labels must be contiguous IDs 0, 1, 2, 3, 4.')
    for modality, matrix in features.items():
        if matrix.shape != (len(labels), FEATURE_DIMENSION):
            raise ValueError(f'{modality} must have shape ({len(labels)}, 384), got {matrix.shape}.')
        if not np.isfinite(matrix).all():
            raise ValueError(f'{modality} contains non-finite values.')
    return sample_ids, labels, features

def make_classifier(seed: int):
    return XGBClassifier(
        objective='multi:softprob',
        num_class=len(CLASS_NAMES),
        n_estimators=N_ESTIMATORS,
        max_depth=MAX_DEPTH,
        learning_rate=LEARNING_RATE,
        subsample=1.0,
        colsample_bytree=1.0,
        min_child_weight=1.0,
        gamma=0.0,
        reg_lambda=1.0,
        reg_alpha=0.0,
        eval_metric='mlogloss',
        random_state=seed,
        n_jobs=N_JOBS,
    )

def aligned_probabilities(classifier, matrix):
    raw = classifier.predict_proba(matrix)
    aligned = np.zeros((len(matrix), len(CLASS_NAMES)), dtype=np.float64)
    for source_column, class_id in enumerate(classifier.classes_.astype(int)):
        aligned[:, class_id] = raw[:, source_column]
    return aligned

In [ ]:
sample_ids, labels, features = load_feature_archive(FEATURE_ARCHIVE)

def load_saved_outer_splits(sample_ids):
    index_by_id = {str(sample_id): index for index, sample_id in enumerate(sample_ids)}
    splits = []
    for fold_number in range(1, OUTER_FOLDS + 1):
        fold_dir = FIXED_FOLD_DIR / f'Fold_{fold_number}'
        train_table = pd.read_excel(fold_dir / 'train.xlsx')
        test_table = pd.read_excel(fold_dir / 'test.xlsx')
        train_ids = train_table[FOLD_ID_COLUMN].astype(str).tolist()
        test_ids = test_table[FOLD_ID_COLUMN].astype(str).tolist()
        missing = [sample_id for sample_id in train_ids + test_ids if sample_id not in index_by_id]
        if missing:
            raise ValueError(f'Fold {fold_number} contains IDs absent from the feature archive: {missing[:3]}')
        train_index = np.asarray([index_by_id[sample_id] for sample_id in train_ids], dtype=int)
        test_index = np.asarray([index_by_id[sample_id] for sample_id in test_ids], dtype=int)
        if np.intersect1d(train_index, test_index).size:
            raise ValueError(f'Fold {fold_number} has overlapping train and test IDs.')
        splits.append((train_index, test_index))
    test_indices = np.concatenate([test_index for _, test_index in splits])
    if len(test_indices) != len(sample_ids) or not np.array_equal(np.sort(test_indices), np.arange(len(sample_ids))):
        raise ValueError('Saved test folds must contain every sample exactly once.')
    return splits

if USE_SAVED_FOLDS:
    outer_splits = load_saved_outer_splits(sample_ids)
else:
    outer_cv = StratifiedKFold(OUTER_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    outer_splits = list(outer_cv.split(np.zeros(len(labels)), labels))
if min(np.bincount(labels)) < OUTER_FOLDS:
    raise ValueError('Every class needs at least five samples for the outer folds.')
print(f'Samples: {len(labels)} | Outer folds: {OUTER_FOLDS} | Inner folds: {INNER_FOLDS}')

In [ ]:
def nested_probabilities(train_x, train_y, test_x):
    if min(np.bincount(train_y)) < INNER_FOLDS:
        raise ValueError('Every class needs at least five samples in each outer-training set.')
    inner_cv = StratifiedKFold(INNER_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    train_prob = np.zeros((len(train_y), len(CLASS_NAMES)), dtype=np.float64)
    test_prob = []
    assigned = np.zeros(len(train_y), dtype=int)
    for inner_number, (fit_index, valid_index) in enumerate(inner_cv.split(train_x, train_y), start=1):
        classifier = make_classifier(RANDOM_STATE + inner_number)
        classifier.fit(
            train_x[fit_index],
            train_y[fit_index],
            sample_weight=compute_sample_weight('balanced', train_y[fit_index]),
        )
        train_prob[valid_index] = aligned_probabilities(classifier, train_x[valid_index])
        test_prob.append(aligned_probabilities(classifier, test_x))
        assigned[valid_index] += 1
    if not np.all(assigned == 1):
        raise RuntimeError('Each outer-training sample must receive one OOF prediction.')
    return train_prob, np.mean(test_prob, axis=0)

PROBABILITY_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for fold_number, (train_index, test_index) in enumerate(outer_splits, start=1):
    train_probabilities = {}
    test_probabilities = {}
    for modality in MODALITIES:
        train_probabilities[modality], test_probabilities[modality] = nested_probabilities(
            features[modality][train_index],
            labels[train_index],
            features[modality][test_index],
        )
    output = {
        'fold': np.asarray(fold_number),
        'train_indices': train_index,
        'test_indices': test_index,
        'train_sample_ids': sample_ids[train_index],
        'test_sample_ids': sample_ids[test_index],
        'y_train': labels[train_index],
        'y_test': labels[test_index],
    }
    for modality in MODALITIES:
        output[f'{modality}_train_probabilities'] = train_probabilities[modality]
        output[f'{modality}_test_probabilities'] = test_probabilities[modality]
    np.savez_compressed(PROBABILITY_OUTPUT_DIR / f'fold_{fold_number:02d}_probabilities.npz', **output)
    print(f'Saved fold {fold_number}')

In [ ]:
metadata = {
    'modalities': list(MODALITIES),
    'class_names': list(CLASS_NAMES),
    'outer_folds': OUTER_FOLDS,
    'inner_folds': INNER_FOLDS,
    'random_state': RANDOM_STATE,
    'class_weighting': 'balanced sample weights in each inner XGBoost fit',
    'xgboost': {
        'objective': 'multi:softprob',
        'n_estimators': N_ESTIMATORS,
        'max_depth': MAX_DEPTH,
        'learning_rate': LEARNING_RATE,
        'subsample': 1.0,
        'colsample_bytree': 1.0,
        'min_child_weight': 1.0,
        'gamma': 0.0,
        'reg_lambda': 1.0,
        'reg_alpha': 0.0,
        'eval_metric': 'mlogloss',
    },
    'outer_split_source': 'saved folds' if USE_SAVED_FOLDS else 'StratifiedKFold(seed=42)',
    'software': {'numpy': np.__version__, 'scikit_learn': sklearn.__version__, 'xgboost': xgboost.__version__},
}
(PROBABILITY_OUTPUT_DIR / 'base_classifier_metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
print('Saved base-classifier metadata.')